# Zadanie 1: dostęp do LLM i mini-wyszukiwarka

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/01-llm-embeddingi/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-01`, w załączniku ten notebook **z wynikami komórek**, bez kluczy API.

**Termin:** przed zajęciami 2.

Części:
1. Dostęp do modelu i wywołanie z wymuszonym formatem odpowiedzi.
2. Koszt w tokenach: polski i angielski.
3. Wyszukiwarka na własnych dokumentach i analiza błędu.

## 0. Konfiguracja

In [ ]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai tiktoken numpy scikit-learn matplotlib seaborn python-dotenv anthropic

Dostawcę wybiera się zmienną `LLM_PROVIDER` (plik `.env` albo w Colabie: ikona klucza → *Secrets*). Instrukcja konfiguracji dostępu: [00-dostep-do-llm](../00-dostep-do-llm/README.md).

| `LLM_PROVIDER` | Wymagania |
|---|---|
| `azure` | `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY` i deploymenty modeli |
| `openai` | `OPENAI_API_KEY` |
| `deepseek` | `DEEPSEEK_API_KEY`; tanie API, bez embeddingów |
| `ollama` | [Ollama](https://ollama.com) lokalnie albo w Colabie na darmowym GPU (komórka niżej), modele `gemma4:e4b`, `gemma3:1b`, `embeddinggemma` |
| `proxy` | uruchomione [proxy dla subskrypcji](../tools/teg-proxy/README.md) (Copilot, Claude, Codex); bez embeddingów |

Wszyscy dostawcy udostępniają API zgodne z OpenAI, więc dalszy kod jest wspólny. Gdy dostawca nie liczy embeddingów, służy do tego `EMBED_PROVIDER` (domyślnie `ollama`).

### Ollama w Colabie (darmowe GPU)

Colab daje za darmo kartę graficzną NVIDIA T4, na której działa Ollama. Najpierw *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*, potem komórka poniżej (odkomentowana). Na początek wystarczy mały `gemma3:1b`; większy `gemma4:e4b` zajmuje ok. 10 GB. Bez niego w *Secrets* ustawia się `OLLAMA_CHAT=gemma3:1b`.

In [ ]:
# Colab z GPU: odkomentuj i uruchom raz na sesję
# !nvidia-smi                                   # czy GPU jest włączone
# !sudo apt-get -qq install -y zstd pciutils > /dev/null
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess
# subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# !ollama pull gemma3:1b
# !ollama pull embeddinggemma
# !ollama pull gemma4:e4b                       # opcjonalnie, ~10 GB

In [ ]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "EMBED_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "DEEPSEEK_API_KEY", "ANTHROPIC_API_KEY"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

# chat: model rozumujący, classic: model bez rozumowania, embed: model embeddingów.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini", "embed": "text-embedding-3-small"},
    "azure":    {"chat": "gpt-5-nano", "classic": "gpt-4o-mini", "embed": "text-embedding-3-small"},
    "deepseek": {"chat": "deepseek-reasoner", "classic": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma3:1b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

## Część 1. Wywołanie modelu

1. Wywołanie `CHAT` z własnym system promptem, który wymusza format odpowiedzi (np. JSON z ustalonymi polami).
2. Wypisanie treści odpowiedzi, `finish_reason` i `usage`.
3. Sprawdzenie w kodzie, czy odpowiedź ma oczekiwany format (np. `json.loads`), i wypisanie wyniku sprawdzenia.

In [ ]:
# TODO: wywołanie modelu z system promptem

## Część 2. Koszt w tokenach

1. Akapit tekstu po polsku (min. 300 znaków) i jego tłumaczenie na angielski.
2. Liczba tokenów obu wersji według tokenizera `o200k_base` (`tiktoken`).
3. Wysłanie obu wersji do modelu z tym samym poleceniem i porównanie `usage.prompt_tokens`.
4. W komórce markdown: o ile procent droższa jest wersja polska i z czego to wynika.

In [ ]:
# TODO: liczenie tokenów i porównanie usage

*Odpowiedź:*

## Część 3. Mini-wyszukiwarka

1. Co najmniej **10 krótkich dokumentów** (1–2 zdania) z jednej, wybranej dziedziny.
2. Embeddingi dokumentów (`embed_client`, model `EMBED`) i funkcja `search(query, k=3)` oparta na cosinusie.
3. **3 pytania**: dwa z poprawną odpowiedzią wyszukiwarki i jedno z błędną.
4. W komórce markdown: przyczyna błędu (2–4 zdania).

In [ ]:
import numpy as np

docs = [
    # TODO: min. 10 dokumentów
]

# TODO: embeddingi dokumentów i funkcja search(query, k=3)

In [ ]:
# TODO: 3 pytania i wyniki

*Wyjaśnienie błędu:*